XGBoost Model with RFECV – Temporal Validation (Training on pre-2022 data and testing on 2022 onward data)

This notebook implements a multi-class xg boost model with recursive feature elimination and cross-validation (RFECV)
trained on data prior to 2022 and evaluated on data from 2022 onward

Classification targets:
- Crohn’s Disease (CD)
- Ulcerative Colitis (UC)
- No IBD

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, classification_report
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import shap
from scipy import stats
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')


import os
import sys

data_path = "../../data"
os.makedirs(data_path, exist_ok=True)

plots_dir = "../../plots"
os.makedirs(plots_dir, exist_ok=True)

savedmodels_dir = "../../saved_models"
os.makedirs(savedmodels_dir, exist_ok=True)
sys.path.append(os.path.abspath(".."))

model = 'XGBoost_rfecv'

## Data Availability and Privacy

Due to patient privacy and institutional restrictions, the original electronic medical record (EMR) dataset used in this study cannot be shared. The dataset included in this repository is a synthetic / de-identified sample provided solely to demonstrate the structure of the data

**Note:** Model training and evaluation results presented in the manuscript were obtained using the original dataset and are not derived from the synthetic data included here.


## Data Preparation

Data extraction, cleaning, and structuring were performed prior to model development to create a structured dataset derived from both structured and unstructured EMR data.
- **Structured data** included demographics, diagnosis codes (ICD-10), medications, laboratory values, and healthcare encounters.
- **Unstructured data** were derived from clinical records (e.g., gastroenterology notes, imaging reports, endoscopy, and pathology reports) using keyword-based extraction to capture clinically relevant terms.


## Train–Test Split (Temporal Validation)

The model was trained using patient data up to 2022 and evaluated on data from 2022 onward.

In [ ]:
pre_df = pd.read_csv(f"{data_path}/IBDregistry_traindata_pre2022.csv")
post_df = pd.read_csv(f"{data_path}/IBDregistry_testdata_post2022.csv")

print("Pre-2022 shape:", pre_df.shape)
print("Post-2022 shape:", post_df.shape)

In [ ]:
pre_raw = pre_df.copy()
post_raw = post_df.copy()

In [ ]:
pre_ids = pre_raw[['PatientDurableKey']].copy()
post_ids = post_raw[['PatientDurableKey']].copy()

### Target Encoding

The categorical target variable (CD, UC, No IBD) was transformed into numeric labels using a label encoder.

In [ ]:
from utils import drop_cols, LABEL_COL

# Encode target
le = LabelEncoder()

y_train = le.fit_transform(pre_raw[LABEL_COL])
y_test  = le.transform(post_raw[LABEL_COL])

X_train = pre_raw.drop(columns=drop_cols + [LABEL_COL])
X_test  = post_raw.drop(columns=drop_cols + [LABEL_COL])

for i, cls in enumerate(le.classes_):
    print(f"{cls} → {i}")

## Preprocessing
- Categorical variables encoded using OneHotEncoder / OrdinalEncoder
- All preprocessing transformations were fit on the training dataset and subsequently applied to the evaluation dataset


In [ ]:
from preprocessing import get_tree_preprocessor

preprocessor =  get_tree_preprocessor(X_train)

In [ ]:
classes = np.arange(len(le.classes_))
y_binarized = label_binarize(y_test, classes=classes)

In [ ]:
from utils import random_state

xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(objective='multi:softprob', num_class = len(le.classes_), random_state=random_state))])

## Model Training
Hyperparameter tuning via GridSearchCV

In [ ]:
from utils import XGgrid_Params

xgb_grid = GridSearchCV(xgb_pipeline, XGgrid_Params, cv=5, scoring="accuracy")

In [ ]:
xgb_grid.fit(X_train, y_train)
print("Best parameters:", xgb_grid.best_params_)
print("Best CV accuracy: {:.4f}".format(xgb_grid.best_score_))

cv_results = xgb_grid.cv_results_
best_idx = xgb_grid.best_index_
split_scores = np.array([
    cv_results[f"split{i}_test_score"][best_idx]
    for i in range(xgb_grid.n_splits_)])
mean_score = split_scores.mean()
std_err = stats.sem(split_scores)
t_value = stats.t.ppf((1 + 0.95) / 2., len(split_scores) - 1)
ci_lower = mean_score - t_value * std_err
ci_upper = mean_score + t_value * std_err
print(f"95% CI: ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"Accuracy - 95% CI: {mean_score:.3f} ({ci_lower:.3f}–{ci_upper:.3f})")


best_model = xgb_grid.best_estimator_
xg_params = {k.replace("model__", ""): v for k, v in xgb_grid.best_params_.items()}


## Feature Selection (RFECV)

Recursive Feature Elimination with Cross-Validation (RFECV) was applied on the training data to select the optimal subset of features based on model performance.

In [ ]:
X_train_processed = best_model.named_steps['preprocess'].transform(X_train)

str_cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=random_state)
selector_final = RFECV(
    estimator=XGBClassifier(**xg_params, objective='multi:softprob', num_class = len(le.classes_), random_state=random_state),
    step=1,
    cv=str_cv,
    scoring="accuracy",
    n_jobs=-1
)

selector_final.fit(X_train_processed, y_train)

In [ ]:
cv_scores = selector_final.cv_results_['mean_test_score']
best_idx = cv_scores.argmax()
best_score = cv_scores[best_idx]
best_features = best_idx + 1

print(f"Best CV Score: {best_score:.4f}")

In [ ]:

feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
selected_features = feature_names[selector_final.support_]
print(f"Optimal number of features: {selector_final.n_features_}")
clean_features = [f.split("__")[-1] for f in selected_features]

cleanfeatures_df = pd.DataFrame({"Selected Features": clean_features})
cleanfeatures_df.to_csv(f"{plots_dir}/pre&post2022_XGBoost_Rfecv_selectedfeatures.csv", index=False)

## Repeated Stratified K-Fold Cross-Validation

To obtain robust and stable performance estimates, we used repeated stratified k-fold cross-validation on the training dataset.
Performance metrics, including AUC, sensitivity, specificity, positive predictive value (PPV), and accuracy, are reported as mean values with corresponding 95% confidence intervals.

In [ ]:
from repeated_stratified_kfold import rfecv_repeated_stratified_kfold

df_results = rfecv_repeated_stratified_kfold(model, selector_final, X_train, y_train, preprocessor, xg_params, le, plots_dir)
df_results

In [ ]:
X_train_selected = selector_final.transform(X_train_processed)
final_model = XGBClassifier(**xg_params, objective='multi:softprob', num_class = len(le.classes_), random_state=random_state)
final_model.fit(X_train_selected, y_train)

In [ ]:
feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
selected_features = feature_names[selector_final.support_]
clean_features = [f.split("__")[-1] for f in selected_features]

importance_values = final_model.feature_importances_

importances_df = pd.DataFrame({
    "Feature": clean_features,
    "Importance": importance_values
})

importances_df = importances_df.sort_values(by="Importance", ascending=False)
importances_df.to_csv(f"{plots_dir}/pre&post2022_XGBoost_Rfecv_importanceresults.csv", index=False)

In [ ]:
from evaluation_plots import shap_plot

feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
selected_features = feature_names[selector_final.support_]
clean_features = [f.split("__")[-1] for f in selected_features]

shap_plot(model, final_model, X_train_selected, clean_features, le, plots_dir)

In [ ]:
X_test_processed = best_model.named_steps['preprocess'].transform(X_test)
X_test_selected = selector_final.transform(X_test_processed)

y_pred = final_model.predict(X_test_selected)
y_prob = final_model.predict_proba(X_test_selected)


In [ ]:
joblib.dump(best_model.named_steps['preprocess'], f"{savedmodels_dir}/pre&post2022_XGBoost_Rfecv_preprocessor.pkl")
joblib.dump(selector_final, f"{savedmodels_dir}/pre&post2022_XGBoost_Rfecv_selector.pkl")
joblib.dump(final_model, f"{savedmodels_dir}/pre&post2022_XGBoost_Rfecv_model.pkl")

In [ ]:
y_pred_original = le.inverse_transform(y_pred)
y_test_original = le.inverse_transform(y_test)

post_ids = post_ids.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

results_df = pd.DataFrame({
    'patient_id': post_ids['PatientDurableKey'],
    'Actual IBD': y_test_original,
    'Predicted IBD': y_pred_original
})

results_df.to_csv(f"{plots_dir}/pre&post2022_XGBoost_Rfecv_predictionresults.csv", index=False)

## Model Evaluation on Test Data
Model performance was evaluated on an independent test dataset (data from 2022 onward), which was not used during model training.

The following metrics were used:
- ROC-AUC
- Confusion matrix
- Sensitivity and specificity

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test,y_pred,target_names=le.classes_))

In [ ]:
from evaluation_plots import display_confusion_matrix, precision_per_class, sensitivity_specificity_per_class,sensitivity_specificity_cduc_noibd, display_roc_curves
cm = display_confusion_matrix(model, y_test, y_pred, le, plots_dir)

In [ ]:
precision_per_class(cm, le)

In [ ]:
sensitivity_specificity_per_class(model,cm,le, plots_dir)

In [ ]:
sensitivity_specificity_cduc_noibd(cm, le)

In [ ]:
auc_macro = roc_auc_score(y_binarized, y_prob, average="macro", multi_class="ovr")
auc_weighted = roc_auc_score(y_binarized, y_prob, average="weighted", multi_class="ovr")

print(f"\nMacro AUC: {auc_macro:.3f}")
print(f"Weighted AUC: {auc_weighted:.3f}")

In [ ]:
display_roc_curves(model, y_test, y_prob, plots_dir, le)